In [ ]:
#The purpose of this program is to find all the places in a
#Charles Dickens story where characters interact with each other

In [ ]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

from nltk.corpus.reader import PlaintextCorpusReader
import csv
import pandas as pd


In [ ]:
import spacy

In [ ]:
import requests

dickensDictionary = {
  "our-mutual-friend": "https://www.gutenberg.org/cache/epub/883/pg883.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt",
  "bleak-house": "https://www.gutenberg.org/cache/epub/1023/pg1023.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt",
  "hard-times": "https://www.gutenberg.org/cache/epub/786/pg786.txt",
  "christmas-carol": "https://www.gutenberg.org/cache/epub/46/pg46.txt",
  "david-copperfield": "https://www.gutenberg.org/cache/epub/766/pg766.txt",
  "tale-of-two-cities": "https://www.gutenberg.org/cache/epub/98/pg98.txt",
  "oliver-twist": "https://www.gutenberg.org/cache/epub/730/pg730.txt",
  "pickwick-papers": "https://www.gutenberg.org/cache/epub/580/pg580.txt",
  "nicholas-nickleby": "https://www.gutenberg.org/cache/epub/967/pg967.txt",
  "old-curiosity-shop": "https://www.gutenberg.org/cache/epub/700/pg700.txt",
  "martin-chuzzlewit": "https://www.gutenberg.org/cache/epub/968/pg968.txt",
  "dombey-and-son": "https://www.gutenberg.org/cache/epub/821/pg821.txt",
  "barnaby-rudge": "https://www.gutenberg.org/cache/epub/917/pg917.txt",
  "american-notes": "https://www.gutenberg.org/cache/epub/675/pg675.txt",
  "sketches-by-boz": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mudfog-papers": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mystery-of-edwin-drood": "https://www.gutenberg.org/cache/epub/564/pg564.txt",
  "little-dorrit": "https://www.gutenberg.org/cache/epub/963/pg963.txt",
  "the-uncommercial-traveller": "https://www.gutenberg.org/cache/epub/914/pg914.txt"
}

In [ ]:
# Open a new file in write-binary mode and write the content of the response to it
## uncommet to download all the books
# for item in dickensDictionary:
#   response = requests.get(dickensDictionary[item])
#   with open(item + '.txt', 'wb') as file:
#     file.write(response.content)

In [ ]:
def get_text(filename):
    # Open the file and read the text
    with open(filename, 'r') as file:  # Replace 'filename.txt' with your actual filename
        text = file.read()
    return text


In [ ]:
text = get_text("../data/little-dorrit.txt")

In [ ]:
def prepare_text(text):
  # tokenize into paragraphs
  paragraphs = text.split("\n\n")
  # remove newlines within paragraphs
  paragraphs = [re.sub(r'[\n]', ' ', doc) for doc in paragraphs]
  return paragraphs

In [ ]:
paragraphs = prepare_text(text)

In [ ]:
paragraphs[0]

In [ ]:
# paragraphs should be a list of paragraphs
def get_tagged_text(paragraphs):
  ## This is meant to be for part of speech tagging
  # Tokenize the text into sentences, then words
  tokens = [word_tokenize(para) for para in paragraphs]
  # Tag the tokens with their part of speech
  pos_tokens = [nltk.pos_tag(tok) for tok in tokens]
  # Chunk the tagged tokens into named entities
  chunked_tokens = [nltk.ne_chunk(tok) for tok in pos_tokens]
  # Convert the trees into IOB tags
  iob_tokens = [tree2conlltags(tok) for tok in chunked_tokens]
  return tokens, pos_tokens, iob_tokens

In [ ]:
nltk.download("punkt")
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

In [ ]:
tokens, pos_tokens, iob_tokens = get_tagged_text(paragraphs)

In [ ]:
iob_tokens[2]

In [ ]:
def get_all_names(iob_tokens):
    allNames = []
    for para in iob_tokens:
        names = []
        current_name = []
        for token, pos, chunk in para:
            if chunk == 'B-PERSON':
                if current_name:
                    # If there's a current name, add it to the list of names
                    names.append(' '.join(current_name))
                # Start a new name
                current_name = [token]
            elif chunk == 'I-PERSON':
                # Continue the current name
                current_name.append(token)
            else:
                if current_name:
                    # If there's a current name, add it to the list of names
                    names.append(' '.join(current_name))
                # Reset the current name
                current_name = []

        # If there's a current name left at the end, add it to the list of names
        if current_name:
            names.append(' '.join(current_name))

        allNames.append(names)
    return allNames

In [ ]:
names_list = get_all_names(iob_tokens)

In [ ]:
names_list


In [ ]:
nlp = spacy.load('en_core_web_sm')

sentence = "Apple is looking at buying U.K. startup for $1 billion"

doc = nlp(paragraphs[12])

for ent in doc.ents:
	print(ent.text, ent.start_char, ent.end_char, ent.label_)

In [ ]:
resultNames = []
for para in paragraphs:
  doc = nlp(para)
  for ent in doc.ents:
    if ent.label_ == "PERSON":
      resultNames.append(ent)

In [ ]:
resultNames

In [ ]:
# Open the file for writing
with open("OMFNames.csv", 'w') as f:
    # Write each word to the file on a new line
    no_repeat = []
    for word in list(set(resultNames)):
        word_1 = str(word)
        if word_1 not in no_repeat:
          no_repeat.append(word_1)
    for word in no_repeat:
      f.write(word + "\n")

In [ ]:
def createInteractionCountList2(interactionsList):
  # Convert the list of tuples into a DataFrame
  df = pd.DataFrame(interactionsList, columns=['Item1', 'Item2', "paraNo", "book"])

  # Count the number of occurrences of each tuple
  df = df.groupby(['Item1', 'Item2']).size().reset_index(name='Count')

  return df

In [ ]:
text = get_text("little-dorrit.txt")
paragraphs = prepare_text(text)
cNamesDf = pd.read_csv("data/CanoniMatchToCanonical9.17.24.csv",encoding="utf-8")
cNameTuples = []
for index, row in cNamesDf.iterrows():
    if row["Canonical Names"] != "?":
      entry = (row["Match Names"], row["Canonical Names"])
      cNameTuples.append(entry)
tokenized_paragraph = [word_tokenize(para) for para in paragraphs]

In [ ]:
## this function gets interactions of raw strings but records canonical names
## all names here should be a list of tuples of raw named to be matched and then canonical name.
def get_interaction_list_with_paraNum_from_canonical_names(paragraphs, cNameTuples, bookName):
    interactionsListWithPara = []
    ## para is a tuple here with the first item being the raw string and the second item the canoncial name
    for index, para in enumerate(paragraphs):
        usedNames = []
        for name in cNameTuples:
          for name2 in cNameTuples:
            if name2 not in usedNames:
              if name[0] in para and name2[0] in para and name[1] != name2[1]:
                interactionsListWithPara.append((name[1], name2[1], index, bookName))
          #usedNames.append(name)
    return interactionsListWithPara

In [ ]:
interactionsList = get_interaction_list_with_paraNum_from_canonical_names(tokenized_paragraph, cNameTuples,  "little-dorrit")

In [ ]:
interactionsList

In [ ]:
paragraphs[6441]

In [ ]:
def save_interactions(interactionsList, filename="interactions.csv"):
    # Open the CSV file in write mode
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)

        # Write the header
        writer.writerow(['Item1', 'Item2', 'Time',"Book"])

        # Write the tuples
        for tuple in interactionsList:
            writer.writerow(tuple)

In [ ]:
save_interactions(interactionsList, "data/Canonical_Names/interactions-canonical-little-dorrit.csv")